In [ ]:
#| default_exp eval_engine

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import numpy as np
from scipy import stats
import structlog
import traceback

# Visualizations
import plotly.express as px
import plotly.graph_objects as go

# Sklearn Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler

log = structlog.get_logger()


In [ ]:
#| export
class FeatureEvaluator:
    """Base class for all feature evaluators. 
    Defines the extraction contract that transforms raw DuckDB queries into 1D arrays."""
    name: str = "base"
    source_file: str = ".dummy.parquet"
    tier: int = 1
    category: str = "general"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        """Transform the loaded raw dataframe into meaningful scalar metrics.
        Called per sample-group or per sample."""
        raise NotImplementedError("Feature evaluators must implement the extract() method.")


In [ ]:
#| export
LABEL_ORDER = ["True ctDNA+", "Possible ctDNA+", "Possible ctDNA−", "Healthy Normal"]
LABEL_COLORS = {
    "True ctDNA+": "#e63946",       # red — highest confidence positive
    "Possible ctDNA+": "#f4a261",   # orange — medium confidence
    "Possible ctDNA−": "#a8dadc",   # light blue — negative
    "Healthy Normal": "#2a9d8f",    # teal — control
}


In [ ]:
#| export
def evaluate_feature(
    feature_values: pd.Series,
    labels: pd.Series,
    total_fragments: pd.Series | None = None,
    max_vaf: pd.Series | None = None,
) -> dict:
    """Run all statistical tests for a single feature in one stratum.
    Outputs metrics directly to scoring dict.
    """
    try:
        groups = {}
        for label in LABEL_ORDER:
            mask = labels == label
            # Ensure index alignment if possible 
            vals = feature_values[mask].dropna()
            groups[label] = vals

        result = {}

        # --- Kruskal-Wallis (4-group) ---
        non_empty = [g for g in groups.values() if len(g) >= 2]
        if len(non_empty) >= 2:
            try:
                kw_stat, kw_p = stats.kruskal(*non_empty)
                result["kw_statistic"] = kw_stat
                result["kw_pvalue"] = kw_p
            except ValueError as e:
                log.warning("kruskal_failed", error=str(e))

        # --- Pairwise Mann-Whitney U ---
        pairs = [
            ("True ctDNA+", "Healthy Normal"),
            ("Possible ctDNA+", "Healthy Normal"),
            ("True ctDNA+", "Possible ctDNA+"),
            ("Possible ctDNA−", "Healthy Normal"),
            ("True ctDNA+", "Possible ctDNA−"),
        ]
        for g1_name, g2_name in pairs:
            g1, g2 = groups.get(g1_name, []), groups.get(g2_name, [])
            key = f"mwu_{g1_name}_vs_{g2_name}".replace(" ", "_").replace("+", "pos").replace("−", "neg")
            if len(g1) >= 2 and len(g2) >= 2:
                try:
                    u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative="two-sided")
                    r = 1 - (2 * u_stat) / (len(g1) * len(g2)) # rank-biserial
                    result[f"{key}_pvalue"] = p_val
                    result[f"{key}_effect_size"] = r
                except ValueError as e:
                    log.debug("mwu_failed", pair=f"{g1_name}-{g2_name}", error=str(e))

        # --- Cohen's d (True ctDNA+ vs Healthy) ---
        true_pos = groups.get("True ctDNA+", pd.Series())
        healthy = groups.get("Healthy Normal", pd.Series())
        if len(true_pos) >= 2 and len(healthy) >= 2:
            pooled_std = np.sqrt((
                (len(true_pos) - 1) * true_pos.std()**2 +
                (len(healthy) - 1) * healthy.std()**2
            ) / (len(true_pos) + len(healthy) - 2))
            if pooled_std > 0:
                result["cohens_d_true_vs_healthy"] = (
                    (true_pos.mean() - healthy.mean()) / pooled_std
                )

        # --- Spearman correlations (Confounders) ---
        if max_vaf is not None:
            valid = feature_values.notna() & max_vaf.notna() & (max_vaf > 0)
            if valid.sum() >= 10:
                try:
                    if np.var(feature_values[valid]) == 0 or np.var(max_vaf[valid]) == 0:
                        r_val, p_val = 0.0, 1.0
                    else:
                        r_val, p_val = stats.spearmanr(feature_values[valid], max_vaf[valid])
                    result["spearman_vaf_r"] = r_val
                    result["spearman_vaf_p"] = p_val
                except ValueError as e:
                    log.warning("spearman_vaf_failed", error=str(e))

        if total_fragments is not None:
            valid = feature_values.notna() & total_fragments.notna()
            if valid.sum() >= 10:
                try:
                    if np.var(feature_values[valid]) == 0 or np.var(total_fragments[valid]) == 0:
                        r_val, p_val = 0.0, 1.0
                    else:
                        r_val, p_val = stats.spearmanr(feature_values[valid], total_fragments[valid])
                    result["spearman_depth_r"] = r_val
                    result["spearman_depth_p"] = p_val
                except ValueError as e:
                    log.warning("spearman_depth_failed", error=str(e))

        # --- Sample counts ---
        for label, vals in groups.items():
            key = label.replace(" ", "_").replace("+", "pos").replace("−", "neg")
            result[f"n_{key}"] = len(vals)

        result["low_power"] = any(len(v) < 10 for v in groups.values())

        return result
    except Exception as e:
        log.error("evaluate_feature_failed", error=str(e), trace=traceback.format_exc())
        return {"error": str(e)}


In [ ]:
#| export
def plot_violin(df: pd.DataFrame, feature_col: str, label_col: str = "label", title: str = "") -> go.Figure:
    """4-group violin with overlaid box plot and individual points for small groups."""
    try:
        fig = px.violin(df, x=label_col, y=feature_col,
                        color=label_col, box=True, points="outliers",
                        category_orders={label_col: LABEL_ORDER},
                        color_discrete_map=LABEL_COLORS,
                        title=title)
        fig.update_layout(showlegend=False, xaxis_title="", yaxis_title=feature_col)
        return fig
    except Exception as e:
        log.error("plot_violin_failed", feature=feature_col, error=str(e))
        return go.Figure()

def plot_density(df: pd.DataFrame, feature_col: str, label_col: str = "label", title: str = "") -> go.Figure:
    """Overlaid density curves per group — shows distribution shape differences."""
    try:
        fig = go.Figure()
        for label in LABEL_ORDER:
            if label not in df[label_col].values:
                continue
            subset = df[df[label_col] == label][feature_col].dropna()
            if len(subset) > 5:
                fig.add_trace(go.Violin(
                    y=subset, name=label, side="positive",
                    line_color=LABEL_COLORS[label], meanline_visible=True,
                    scalemode="width", width=0.8,
                ))
        fig.update_layout(title=title, violinmode="overlay")
        return fig
    except Exception as e:
        log.error("plot_density_failed", feature=feature_col, error=str(e))
        return go.Figure()

def plot_feature_vs_vaf(df: pd.DataFrame, feature_col: str, vaf_col: str = "max_vaf", label_col: str = "label", title: str = "") -> go.Figure:
    """Continuous relationship between feature and tumor burden (VAF proxy)."""
    try:
        cancer = df[df[vaf_col] > 0]  # exclude healthy & zero-VAF
        if cancer.empty:
            return go.Figure()
        fig = px.scatter(cancer, x=vaf_col, y=feature_col, color=label_col,
                         color_discrete_map=LABEL_COLORS, opacity=0.5,
                         trendline="lowess", log_x=True,
                         title=title)
        return fig
    except Exception as e:
        log.error("plot_vs_vaf_failed", feature=feature_col, error=str(e))
        return go.Figure()

def plot_roc_curves(y_true_dict: dict[str, np.ndarray],
                    y_score_dict: dict[str, np.ndarray], title: str = "") -> go.Figure:
    """Overlay ROC curves for multiple comparisons."""
    try:
        fig = go.Figure()
        for name, y_true in y_true_dict.items():
            if name not in y_score_dict:
                continue
            y_score = y_score_dict[name]
            if len(np.unique(y_true)) < 2:
                continue # Cannot plot ROC with 1 class
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            fig.add_trace(go.Scatter(
                x=fpr, y=tpr, name=f"{name} (AUC={roc_auc:.3f})",
                mode="lines",
            ))
        fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                                 line=dict(dash="dash", color="gray"), name="Random"))
        fig.update_layout(title=title, xaxis_title="False Positive Rate", yaxis_title="True Positive Rate")
        return fig
    except Exception as e:
        log.error("plot_roc_curves_failed", error=str(e))
        return go.Figure()

def plot_feature_importance(importances: dict[str, float], title: str = "") -> go.Figure:
    """Bar plot of RF feature importances."""
    try:
        df = pd.DataFrame({"feature": list(importances.keys()),
                           "importance": list(importances.values())})
        df = df.sort_values("importance", ascending=True).tail(20)  # Top 20
        fig = px.bar(df, x="importance", y="feature", orientation="h", title=title)
        return fig
    except Exception as e:
        log.error("plot_rf_importance_failed", error=str(e))
        return go.Figure()

def plot_threshold_sensitivity(results_df: pd.DataFrame, title: str = "") -> go.Figure:
    """Show how label counts shift with VAF/min_variants thresholds."""
    try:
        fig = px.line(results_df, x="min_vaf", y="Possible ctDNA+",
                      color="min_variants", markers=True, title=title)
        return fig
    except Exception as e:
        log.error("plot_threshold_sen_failed", error=str(e))
        return go.Figure()


In [ ]:
#| export
def single_feature_model(
    X: np.ndarray,            # shape (n_samples, n_sub_metrics)
    y: np.ndarray,            # binary labels (1 = positive class)
    cancer_types: np.ndarray | None = None,
    n_folds: int = 5,
    random_state: int = 42,
) -> tuple[dict, object, object, object]:
    """Train LR, RF, and XGB on a single feature's sub-metrics.
    Returns (results_dict, lr_model, rf_model, xgb_model).
    """
    import time
    from sklearn.metrics import classification_report, confusion_matrix
    try:
        from xgboost import XGBClassifier
    except ImportError:
        XGBClassifier = None

    results = {}
    try:
        y = y.astype(int)
        if len(np.unique(y)) < 2:
            log.warning("single_class_y", y_unique=np.unique(y))
            return {"error": "Only one class present in target array"}, None, None, None
        
        # Guard for small groups where n_splits doesn't work
        min_class_counts = np.bincount(y).min()
        folds = min(n_folds, min_class_counts)
        if folds < 2:
            return {"error": "Not enough samples per class for Stratified CV (<2)"}, None, None, None
            
        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
        
        def get_optimal_threshold(y_true, y_pred_prob):
            try:
                fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
                if len(thresholds) > 0:
                    optimal_idx = np.argmax(tpr - fpr)
                    return float(thresholds[optimal_idx])
            except: pass
            return 0.5

        def safely_get_metrics(y_true, y_pred_prob, threshold=0.5):
            y_pred = (y_pred_prob >= threshold).astype(int)
            rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
            cm = confusion_matrix(y_true, y_pred).tolist()
            return rep, cm
        
        # --- Logistic Regression ---
        lr = LogisticRegression(max_iter=1000, random_state=random_state)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        lr_probs = cross_val_predict(lr, X_scaled, y, cv=cv, method="predict_proba")[:, 1]
        results["auc_lr"] = roc_auc_score(y, lr_probs)
        lr_opt = get_optimal_threshold(y, lr_probs)
        results["lr_optimal_threshold"] = lr_opt
        lr_rep, lr_cm = safely_get_metrics(y, lr_probs, threshold=lr_opt)
        results["lr_classification_report"] = lr_rep
        results["lr_confusion_matrix"] = lr_cm
        
        t0 = time.perf_counter()
        lr.fit(X_scaled, y)
        results["lr_training_time_sec"] = time.perf_counter() - t0
        results["lr_coef_direction"] = "positive" if lr.coef_[0].mean() > 0 else "negative"
        
        # --- Random Forest ---
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=5,
            min_samples_leaf=max(1, min(10, len(y)//10)), 
            random_state=random_state,
            class_weight="balanced", 
        )
        rf_probs = cross_val_predict(rf, X, y, cv=cv, method="predict_proba")[:, 1]
        results["auc_rf"] = roc_auc_score(y, rf_probs)
        rf_opt = get_optimal_threshold(y, rf_probs)
        results["rf_optimal_threshold"] = rf_opt
        rf_rep, rf_cm = safely_get_metrics(y, rf_probs, threshold=rf_opt)
        results["rf_classification_report"] = rf_rep
        results["rf_confusion_matrix"] = rf_cm
        
        t0 = time.perf_counter()
        rf.fit(X, y)
        results["rf_training_time_sec"] = time.perf_counter() - t0
        results["rf_feature_importances"] = rf.feature_importances_.tolist()
        
        # --- XGBoost ---
        xgb = None
        if XGBClassifier is not None:
            xgb = XGBClassifier(
                n_estimators=100, max_depth=5,
                learning_rate=0.1, random_state=random_state,
                eval_metric="logloss",
                use_label_encoder=False
            )
            try:
                xgb_probs = cross_val_predict(xgb, X, y, cv=cv, method="predict_proba")[:, 1]
                results["auc_xgb"] = roc_auc_score(y, xgb_probs)
                xgb_opt = get_optimal_threshold(y, xgb_probs)
                results["xgb_optimal_threshold"] = xgb_opt
                xgb_rep, xgb_cm = safely_get_metrics(y, xgb_probs, threshold=xgb_opt)
                results["xgb_classification_report"] = xgb_rep
                results["xgb_confusion_matrix"] = xgb_cm
                
                t0 = time.perf_counter()
                xgb.fit(X, y)
                results["xgb_training_time_sec"] = time.perf_counter() - t0
            except Exception as e:
                log.warning("xgboost_cv_failed", error=str(e))
                xgb = None
        
        # Diff Delta
        results["auc_delta_rf_lr"] = results["auc_rf"] - results["auc_lr"]
        if "auc_xgb" in results:
            results["auc_delta_xgb_rf"] = results["auc_xgb"] - results["auc_rf"]
            
        # --- Cancer Type Subgroup Analysis (Top 10) ---
        if cancer_types is not None:
            import pandas as pd
            c_stats = []
            c_series = pd.Series(cancer_types)
            top_10 = c_series.value_counts().nlargest(10).index
            
            for c_type in top_10:
                mask = (cancer_types == c_type)
                if mask.sum() < 5: continue # Skip if too small
                y_sub = y[mask]
                
                # We need at least one positive and one negative for AUC
                if len(np.unique(y_sub)) > 1:
                    rf_prob_sub = rf_probs[mask]
                    sub_auc = roc_auc_score(y_sub, rf_prob_sub)
                else:
                    sub_auc = None
                    
                # True Positive Rate (Sensitivity) at optimal threshold
                pos_mask = (y_sub == 1)
                if pos_mask.sum() > 0:
                    rf_pred_pos = (rf_probs[mask][pos_mask] >= rf_opt).astype(int)
                    sub_tpr = rf_pred_pos.sum() / len(rf_pred_pos)
                else:
                    sub_tpr = None
                    
                c_stats.append({
                    "cancer_type": c_type,
                    "n_samples": int(mask.sum()),
                    "n_positives": int(pos_mask.sum()),
                    "rf_auc": sub_auc,
                    "rf_sensitivity": sub_tpr
                })
            results["cancer_type_stats"] = c_stats
            
        return results, lr, rf, xgb
    except Exception as e:
        import traceback
        log.error("single_feature_model_failed", error=str(e), trace=traceback.format_exc())
        return {"error": str(e)}, None, None, None